# 01_text_prep

This notebook prepares CFPB credit card complaint narratives for downstream NLP work.

Main tasks:
- Load the processed complaint extract created in `00_setup.ipynb`.
- Inspect the raw complaint narratives.
- Apply a preliminary text cleaning function.
- Create basic text length fields.
- Save a text-prepped dataset for the next notebook.

This notebook keeps the original complaint narrative intact and adds cleaned text fields rather than overwriting the source text.

### Imports

In [31]:
# Standard library
import re
import unicodedata
from pathlib import Path

#Third-party libraries
import pandas as pd

### Paths

In [32]:
# Assume the notebook is being run from the notebooks/ folder.
# PROJECT_ROOT points to the repo root so file paths stay portable.
PROJECT_ROOT = Path.cwd().parent

# Processed dataset folder used for transferring between notebooks.
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

PROJECT_ROOT, DATA_PROCESSED

(WindowsPath('c:/Users/tivon/postgres-lab/python-cfpb-complaint-nlp'),
 WindowsPath('c:/Users/tivon/postgres-lab/python-cfpb-complaint-nlp/data/processed'))

### Load the setup output

In [33]:
# Load the prepared complaint data extract produced in 00_setup.ipynb.
input_csv = DATA_PROCESSED / "cfpb_credit_card_complaints_python_input.csv"

complaints = pd.read_csv(input_csv, parse_dates=["date_received", "quarter_start"])

complaints.shape

(76879, 14)

### Quick preview

In [34]:
# Preview the key columns and a few narratives before cleaning.
complaints[
    [
        "complaint_id",
        "date_received",
        "quarter_label",
        "issue",
        "sub_issue",
        "consumer_complaint_narrative",
    ]
].head()

,complaint_id,date_received,quarter_label,issue,sub_issue,consumer_complaint_narrative
0,8085192,2024-01-01,2024 Q1,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,In XXXXXXXX XXXX XXXX I made a payment to BOA...
1,8085289,2024-01-01,2024 Q1,Problem when making payments,Problem during payment process,I had submitted for an electronic payment to b...
2,8085300,2024-01-01,2024 Q1,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,This is regarding my credit card issued by Gol...
3,8085634,2024-01-01,2024 Q1,Getting a credit card,Sent card you never applied for,Received card never applied for Some current c...
4,8085685,2024-01-01,2024 Q1,"Advertising and marketing, including promotion...",Didn't receive advertised or promotional terms,"On the XXXX of XXXX, I upgraded my cobranded C..."


### Inspect a few narrative more clearly


In [35]:
# Print a few complaint narratives in full to inspect formatting issues such as line breaks, repeated spaces, unusual characters, or redactions.
sample_narratives = complaints["consumer_complaint_narrative"].head(3)

for i, text in enumerate(sample_narratives, start=1):
    print(f"\n--- Narrative {i} ---\n")
    print(text[:2000])


--- Narrative 1 ---

In XXXXXXXX XXXX XXXX  I made a payment to BOA, where I had 2 credit cards. I made the payment to the wrong card, but it had a XXXX. The bank put the payment to the correct card and sent me a letter telling me they had done this and I wouldn't be charged late fees etc. On my credit report I have a late payment showing, which has made my other cards raise my apr 's. I called the company and was told it would take time to come off my report. it didn't. I called again and was told that the late payment couldn't be taken off. I sent a dispute through XXXX several times and it is still on my report. My statements for XXXX, XXXX, and XXXX from BOA show no fees charged so why can't we get this removed from my report.

--- Narrative 2 ---

I had submitted for an electronic payment to be processed on XX/XX/ for {$1100.00}. I learned on XX/XX/ that they reversed the payment with reason " payment stopped. '' They charged me {$40.00} to the account. I requested reason why and

### Define the cleaning function

In [36]:
# Create a conservative text cleaning function.
# The goal is to standardize the narrative text for analysis without stripping away too much meaning in the first pass.
def clean_complaint_text(text: str) -> str:
    """Clean a CFPB complaint narrative for downstream NLP use."""
    if pd.isna(text):
        return ""
    
    # Normalize unicodedata.normalize("NFKC", text)
    text = unicodedata.normalize("NFKC", text)

    # Convert to lowercase for easier grouping and token analysis later.
    text = text.lower()

    # Remove long redaction masks such as xxxx or xxxxxxxx.
    text = re.sub(r"\bx{4,}\b", " ", text)

    # Remove short redaction masks such as xx and repeated xx fragments.
    text = re.sub(r"\bxx\b", " ", text)
    text = re.sub(r"\b(?:xx\s+){1,}xx\b", " ", text)

    # Remove masked date-like patterns such as xx/xx/xxxx.
    text = re.sub(r"\b[x]{1,2}/[x]{1,2}/[x]{2,4}\b", " ", text)

    # Remove masked numeric placeholders such as 00 when they appear as standalone tokens.
    text = re.sub(r"\b00\b", " ", text)

    # Remove masked age/date fragments such as "xx year" or "xx years".
    text = re.sub(r"\bxx\s+years?\b", " ", text)
    text = re.sub(r"\byears?\s+old\b", " years old ", text)

    # Replace line breaks and tabs with spaces.
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove extra whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    return text

### Create cleaned text columns

In [37]:
# Preserve the original narrative text and add cleaned text fields for NLP.
complaints["narrative_text_clean"] = complaints["consumer_complaint_narrative"].apply(clean_complaint_text)

# Add simple length fields for later quality checks and exploratory analysis.
complaints["narrative_char_count"] = complaints["narrative_text_clean"].str.len()
complaints["narrative_word_count"] = complaints["narrative_text_clean"].str.split().str.len()

complaints[
    [
        "consumer_complaint_narrative",
        "narrative_text_clean",
        "narrative_char_count",
        "narrative_word_count",
    ]
].head()

,consumer_complaint_narrative,narrative_text_clean,narrative_char_count,narrative_word_count
0,In XXXXXXXX XXXX XXXX I made a payment to BOA...,"in i made a payment to boa, where i had 2 cred...",677,142
1,I had submitted for an electronic payment to b...,i had submitted for an electronic payment to b...,798,154
2,This is regarding my credit card issued by Gol...,this is regarding my credit card issued by gol...,881,162
3,Received card never applied for Some current c...,received card never applied for some current c...,61,10
4,"On the XXXX of XXXX, I upgraded my cobranded C...","on the of , i upgraded my cobranded chase ) cr...",407,78


### Basic text checks

In [38]:
# Confirm that the cleaned text column is populated and inspect length ranges.
print("Rows:", len(complaints))
print("Null cleaned text:", complaints["narrative_text_clean"].isna().sum())
print("Blank cleaned text:", (complaints["narrative_text_clean"] == "").sum())
print("Min words:", complaints["narrative_word_count"].min())
print("Median words:", complaints["narrative_word_count"].median())
print("Max words:", complaints["narrative_word_count"].max())

Rows: 76879
Null cleaned text: 0
Blank cleaned text: 1
Min words: 0
Median words: 152.0
Max words: 6263


### Short narratives check

In [39]:
# Review the shortest cleaned narratives to see whether any records look too sparse to be useful for later NLP steps.
complaints.sort_values("narrative_word_count")[
    [
        "complaint_id",
        "issue",
        "sub_issue",
        "narrative_word_count",
        "narrative_text_clean",
    ]
].head(10)

,complaint_id,issue,sub_issue,narrative_word_count,narrative_text_clean
52628,13978559,Problem with a purchase shown on your statement,Credit card company isn't resolving a dispute ...,0,
39233,11854296,Getting a credit card,Card opened without my consent or knowledge,1,#name?
4370,8354948,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,2,unknown charges
42457,12276105,Improper use of your report,Credit inquiries on your report that you don't...,2,/ /
76242,18321019,Getting a credit card,Card opened without my consent or knowledge,2,/ /
5997,8463669,Incorrect information on your report,Information is missing that should be on the r...,3,reporting inaccurate infomation
10025,8704779,Problem with a company's investigation into an...,Was not notified of investigation status or re...,4,see the attached documents.
15087,9030620,Problem with a company's investigation into an...,Was not notified of investigation status or re...,4,see the attached documents.
13291,8914926,Incorrect information on your report,Account information incorrect,4,account pay by insurance.
8882,8644435,Problem with a company's investigation into an...,Was not notified of investigation status or re...,4,see the attached documents.


### Save the text-prepped file

In [40]:
# Save the text-prepped dataset for the theme-analysis notebook.
output_csv = DATA_PROCESSED / "cfpb_credit_card_complaints_text_prep.csv"

complaints.to_csv(output_csv, index=False)

output_csv

WindowsPath('c:/Users/tivon/postgres-lab/python-cfpb-complaint-nlp/data/processed/cfpb_credit_card_complaints_text_prep.csv')

### Outcome

The complaint narratives were:
- loaded from the extract
- reviewed for basic formatting issues
- cleaned into a standardized text field
- saved for the next stage of NLP analysis.